In [31]:
import gymnasium as gym
import numpy as np
import torch
from torch import nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torch.distributions import Categorical
from tqdm import tqdm
from torch.optim import Adam
import time

In [88]:
class replay_buffer:
    ## THis replay buffer is not perfectly coded but at least we do not use cumbersome datastructures....
    ##At least I tried, in the end I did (Arthur Morgar RIP)
    def __init__(self, capacity:int = 1000):
        self.capacity = capacity
        self.sinit()
        self.enumerate = [i for i in range(capacity)]
        self.first_sweep = False
        self.counter = 0
        
        
    def reset(self)->None:
        self.sinit()
        
    def append(self, state, action, next_state, reward, terminated, truncated)->None:
        ## zero counter if there is no room
        if self.counter >= self.capacity:
            self.first_sweep = True
            self.counter = 0
        self.state[self.counter] = state
        self.next_state[self.counter] = next_state
        self.reward[self.counter] = reward
        self.action[self.counter] = action
        self.terminated[self.counter] = terminated
        self.truncated[self.counter] = truncated
        
        
        ## update counter
        self.counter += 1

    def sinit(self)->None:
        self.state = np.empty((self.capacity, 8), dtype = np.float32)
        self.action = np.empty(self.capacity, dtype = np.float32)
        self.next_state = np.empty((self.capacity, 8), dtype = np.float32)
        self.reward = np.empty(self.capacity, dtype = np.float32)
        self.terminated = np.empty(self.capacity, dtype = np.bool_)
        self.truncated = np.empty(self.capacity, dtype = np.bool_)
    
    
    def sample_batch(self, size:int = 100, p: np.ndarray = None)->tuple[np.ndarray]:       
        
        if not self.first_sweep:
            indexes = np.random.choice([i for i in range(self.counter)], size = min(self.counter, size), p = p)
        else:
            size = min(self.counter, size)
            indexes = np.random.choice(self.enumerate, size = size, p = p, replace = False)
            
        return map(lambda x: x[indexes], [self.state, 
                                          self.action, 
                                          self.next_state, 
                                          self.reward, 
                                          self.terminated, 
                                          self.truncated])

In [207]:
Q = nn.Sequential(*[nn.Linear(8, 30), 
                          nn.ReLU(),
                          nn.Linear(30, 40),
                          nn.ReLU(),
                          nn.Linear(40, 4),
                         ]); ### This dude is the Q function
Q_old = nn.Sequential(*[nn.Linear(8, 30), 
                          nn.ReLU(),
                          nn.Linear(30, 40),
                          nn.ReLU(),
                          nn.Linear(40, 4),
                         ])
#Q_old.load_state_dict(Q.state_dict())

@torch.no_grad  ## This prick is 
def choose_action(state:np.ndarray, 
                  network:nn.Module = Q, 
                 )->torch.tensor:
    softmaxed_logits = torch.softmax(network(torch.tensor(state, dtype = torch.float32)), -1)
    probs = Categorical(softmaxed_logits).sample()
    return probs, softmaxed_logits
@torch.compile
@torch.no_grad
def update(Q, Q_old, α = 0.1):
    for param_Q, param_Q_old in zip(Q.parameters(), Q_old.parameters()):
        param_Q_old += α*(param_Q-param_Q_old)        

In [236]:
env = gym.make("LunarLander-v2", 
               gravity = -10.0, 
               enable_wind = True, 
               wind_power = 10.0,
               turbulence_power = 1.5,
               #render_mode ="human"
              )
γ = 0.99
ϵ = 1.0
opt = Adam(Q.parameters(), 0.0001)

buffer = replay_buffer(100000)

cum_rew_arr = []

iterator = tqdm(range(5))
for episode in iterator:
    
    last_state, info = env.reset()
    cum_rew = 0
    
    for i in range(5000):
        
        action = choose_action(last_state, Q)[0].item()
        
        current_state, reward, terminated, truncated, info = env.step(action)
        
        buffer.append(last_state, action,current_state, reward, terminated, truncated)
    
        last_state = current_state

        Q.train()
    
        last_state_, action_, current_state_, reward_, terminated_, truncated_ = map(lambda x: torch.tensor(x), 
                                                                                     buffer.sample_batch(800))
       
        
        a = action_.type(torch.int64).view(-1,1)
    
        with torch.no_grad():
        ## Expected sarsa we are doing here!!!!
            #sarsa term
            avg = (Q(current_state_).softmax(-1)*Q(current_state_)).sum(-1)
            
            δ = reward_ + γ*(1-1*(terminated_ + truncated_))*avg
       
        opt.zero_grad()
    
        z = ((δ - Q(last_state_).gather(1, a).squeeze())**2).mean()
    
        z.backward()
        opt.step()
        
        cum_rew = 0.1*cum_rew + (1-0.1)*reward_.mean().item()
        
        if terminated or truncated:
            last_state, info = env.reset()
            break
    update(Q, Q_old, α = 0.1)
     
    iterator.set_description(f"{cum_rew}")
    cum_rew_arr.append(cum_rew)
env.close()

0.3868609621918174: 100%|█████████████████████████| 5/5 [00:01<00:00,  2.75it/s]


In [275]:
env = gym.make("LunarLander-v2", 
                max_episode_steps = 700,  
                gravity = -10.0, 
                enable_wind = True,
                wind_power =10.0, 
                turbulence_power = 1.5,                          
               render_mode = "human"
                )

state, info = env.reset()
γ = 0.99

for c in range(500):
    Q.eval()
    action, _ = choose_action(state, Q)  # agent policy that uses the observation and info
    state, reward, terminated, truncated, info = env.step(action.item())
    if terminated or truncated:
        state, info = env.reset()
        break
env.close()